# LeetCode 1638 — Count Substrings That Differ by One Character bằng Python

Link đề: https://leetcode.com/problems/count-substrings-that-differ-by-one-character/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Dynamic Programming, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def countSubstrings(self, s: str, t: str) -> int:
        ...
```

Notebook đi từ **vét cạn** (so mọi cặp xâu con) → **mở rộng dần** từ mỗi cặp điểm đầu → **lấy ký tự khác nhau làm tâm** rồi loang hai phía → **quy hoạch động** hai bảng `f`, `g` → **duyệt theo đường chéo** chỉ với `O(1)` bộ nhớ.

## 1. Mục tiêu học

1. Hiểu đúng đề: đếm số **cặp** (xâu con của `s`, xâu con của `t`) **cùng độ dài**, khác nhau **đúng một** vị trí — đếm theo **vị trí**, không phải theo nội dung khác nhau.
2. Biết biến một bài "so cặp xâu con" thành bài **trên đường chéo** của lưới `m × n`: cặp `(s[i..i+L), t[j..j+L))` chạy dọc đường chéo `i − j = const`.
3. Nắm kỹ thuật "**lấy chỗ khác nhau làm tâm**": số cách = `(số ký tự bằng nhau liên tiếp bên trái + 1) × (bên phải + 1)`.
4. Cài đặt quy hoạch động độ dài **hậu tố chung** `f[i][j]` và **tiền tố chung** `g[i][j]`, rồi tối ưu xuống `O(1)` bộ nhớ bằng duyệt đường chéo.
5. Đối chiếu nhiều cách với brute force trên dữ liệu ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho hai xâu `s` và `t`. Hãy đếm số cách chọn một **xâu con khác rỗng** của `s` rồi **thay đúng một ký tự** của nó bằng một ký tự **khác**, sao cho xâu thu được là một xâu con của `t`.

Nói cách khác: đếm số cặp (xâu con của `s`, xâu con của `t`) **cùng độ dài** và **khác nhau đúng 1 ký tự**. Hai xâu con khác vị trí được tính là khác nhau, kể cả khi nội dung giống nhau.

Ví dụ: `"compute"` (trong `"computer"`) và `"computa"` (trong `"computation"`) chỉ khác ở `'e'`/`'a'` ⇒ là một cặp hợp lệ.

**Ví dụ 1:** `s = "aba"`, `t = "baba"` → `6`. Các cặp:
- `("a"` ở vị trí 0 của s, `"b"` ở vị trí 0 của t`)`, `("a"`@0, `"b"`@2`)`
- `("a"`@2, `"b"`@0`)`, `("a"`@2, `"b"`@2`)`
- `("b"`@1, `"a"`@1`)`, `("b"`@1, `"a"`@3`)`

**Ví dụ 2:** `s = "ab"`, `t = "bb"` → `3`. Các cặp: `("a"@0, "b"@0)`, `("a"@0, "b"@1)`, `("ab", "bb")`.

**Ràng buộc:**
- `1 ≤ s.length, t.length ≤ 100`
- `s`, `t` chỉ gồm chữ cái tiếng Anh in thường.

In [1]:
import random
import time
from typing import List

examples = [("aba", "baba", 6), ("ab", "bb", 3)]

def list_pairs(s, t):
    """Liệt kê mọi cặp (i, j, L) mà s[i:i+L] và t[j:j+L] khác đúng 1 ký tự."""
    out = []
    for i in range(len(s)):
        for j in range(len(t)):
            for L in range(1, min(len(s) - i, len(t) - j) + 1):
                diff = sum(a != b for a, b in zip(s[i:i + L], t[j:j + L]))
                if diff == 1:
                    out.append((i, j, L, s[i:i + L], t[j:j + L]))
    return out

for s, t, exp in examples:
    pairs = list_pairs(s, t)
    print(f"s={s!r}, t={t!r} -> {len(pairs)} cặp (kỳ vọng {exp})")
    for i, j, L, a, b in pairs:
        print(f"    s[{i}:{i + L}]={a!r:6} t[{j}:{j + L}]={b!r}")

s='aba', t='baba' -> 6 cặp (kỳ vọng 6)
    s[0:1]='a'    t[0:1]='b'
    s[0:1]='a'    t[2:3]='b'
    s[1:2]='b'    t[1:2]='a'
    s[1:2]='b'    t[3:4]='a'
    s[2:3]='a'    t[0:1]='b'
    s[2:3]='a'    t[2:3]='b'
s='ab', t='bb' -> 3 cặp (kỳ vọng 3)
    s[0:1]='a'    t[0:1]='b'
    s[0:2]='ab'   t[0:2]='bb'
    s[0:1]='a'    t[1:2]='b'


## 3. Kiến thức nền

### 3.1 Xâu con và cắt lát

Xâu con liên tục `s[i:i+L]` bắt đầu ở `i`, dài `L`. Xâu độ dài `m` có `m(m+1)/2` xâu con (tính theo vị trí). Cắt lát tạo xâu mới, tốn `O(L)`.

In [2]:
s = "abc"
subs = [(i, s[i:j]) for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
print(subs)
print("số xâu con =", len(subs), "= m(m+1)/2 =", len(s) * (len(s) + 1) // 2)

[(0, 'a'), (0, 'ab'), (0, 'abc'), (1, 'b'), (1, 'bc'), (2, 'c')]
số xâu con = 6 = m(m+1)/2 = 6


### 3.2 `zip` và đếm số vị trí khác nhau (khoảng cách Hamming)

`zip(a, b)` ghép từng cặp ký tự cùng vị trí. `sum(x != y for x, y in zip(a, b))` đếm số vị trí khác nhau — gọi là **khoảng cách Hamming** của hai xâu cùng độ dài. `True` được cộng như `1`.

In [3]:
def hamming(a, b):
    return sum(x != y for x, y in zip(a, b))

print(hamming("compute", "computa"))
print(hamming("abc", "abc"), hamming("abc", "xbz"))
print(list(zip("ab", "bb")))

1
0 2
[('a', 'b'), ('b', 'b')]


### 3.3 Mảng 2 chiều trong Python

Tạo bảng `(m+1) × (n+1)` toàn số 0: `[[0] * (n + 1) for _ in range(m + 1)]`. **Không** dùng `[[0] * (n+1)] * (m+1)` vì các hàng sẽ là **cùng một list** (sửa một hàng là sửa tất cả).

In [4]:
good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][0] = 1
bad[0][0] = 1
print("đúng:", good)
print("sai :", bad, "<- hàng 2 cũng bị đổi")

đúng: [[1, 0, 0], [0, 0, 0]]
sai : [[1, 0, 0], [1, 0, 0]] <- hàng 2 cũng bị đổi


### 3.4 Đường chéo của lưới chỉ số

Đặt `i` (chỉ số trong `s`) theo hàng, `j` (chỉ số trong `t`) theo cột. Một cặp xâu con `s[i..i+L)` và `t[j..j+L)` tương ứng với các ô `(i, j), (i+1, j+1), …, (i+L−1, j+L−1)` — một **đoạn liên tục trên đường chéo** `i − j = d`. Có `m + n − 1` đường chéo; mỗi đường chéo bắt đầu ở hàng 0 hoặc cột 0.

In [5]:
m, n = 3, 4
starts = [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]
for i0, j0 in starts:
    cells = []
    i, j = i0, j0
    while i < m and j < n:
        cells.append((i, j)); i += 1; j += 1
    print(f"đường chéo d = i - j = {i0 - j0:>2}: {cells}")

đường chéo d = i - j =  0: [(0, 0), (1, 1), (2, 2)]
đường chéo d = i - j = -1: [(0, 1), (1, 2), (2, 3)]
đường chéo d = i - j = -2: [(0, 2), (1, 3)]
đường chéo d = i - j = -3: [(0, 3)]
đường chéo d = i - j =  1: [(1, 0), (2, 1)]
đường chéo d = i - j =  2: [(2, 0)]


## 4. Ý tưởng nền tảng / trực giác

Xét `s = "ab"`, `t = "bb"`. Vẽ bảng so sánh `s[i] == t[j]` (dấu `=` nếu bằng, `x` nếu khác):

| | t[0]='b' | t[1]='b' |
|---|---|---|
| s[0]='a' | x | x |
| s[1]='b' | = | = |

Một cặp xâu con = một **đoạn liên tục trên đường chéo**. Cặp hợp lệ ⇔ đoạn đó chứa **đúng một** dấu `x`.
- Đường chéo `d=0`: ô `(0,0)=x`, `(1,1)==`. Các đoạn: `[x]` ✔, `[=]` ✘, `[x,=]` ✔ ⇒ 2.
- Đường chéo `d=−1`: ô `(0,1)=x` ⇒ `[x]` ✔ ⇒ 1.
- Đường chéo `d=1`: ô `(1,0)==` ⇒ 0.

Tổng `3` — khớp đề. Bài toán quy về: **trên mỗi dãy ký hiệu `=`/`x`, đếm số đoạn con chứa đúng một `x`**.

Với một `x` cố định, đoạn chứa nó và chỉ nó có thể kéo **trái** qua các `=` liền kề (`l` ô) và **phải** qua các `=` liền kề (`r` ô): có `(l + 1)(r + 1)` lựa chọn.

In [6]:
def diag_symbols(s, t, i0, j0):
    out = []
    i, j = i0, j0
    while i < len(s) and j < len(t):
        out.append('=' if s[i] == t[j] else 'x'); i += 1; j += 1
    return ''.join(out)

s, t = "aba", "baba"
total = 0
for i0, j0 in [(0, j) for j in range(len(t))] + [(i, 0) for i in range(1, len(s))]:
    sym = diag_symbols(s, t, i0, j0)
    cnt = sum(sym[a:b].count('x') == 1 for a in range(len(sym)) for b in range(a + 1, len(sym) + 1))
    total += cnt
    print(f"bắt đầu ({i0},{j0}): {sym:<4} -> {cnt} đoạn có đúng 1 'x'")
print("Tổng:", total)

bắt đầu (0,0): xxx  -> 3 đoạn có đúng 1 'x'
bắt đầu (0,1): ===  -> 0 đoạn có đúng 1 'x'
bắt đầu (0,2): xx   -> 2 đoạn có đúng 1 'x'
bắt đầu (0,3): =    -> 0 đoạn có đúng 1 'x'
bắt đầu (1,0): ==   -> 0 đoạn có đúng 1 'x'
bắt đầu (2,0): x    -> 1 đoạn có đúng 1 'x'
Tổng: 6


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi điểm đầu `i` trong `s`, mọi điểm đầu `j` trong `t`, mọi độ dài `L`; so sánh trực tiếp hai xâu con và đếm khi khoảng cách Hamming bằng 1.

In [7]:
def count_brute(s, t):
    m, n = len(s), len(t)
    ans = 0
    for i in range(m):
        for j in range(n):
            for L in range(1, min(m - i, n - j) + 1):
                diff = 0
                for k in range(L):
                    if s[i + k] != t[j + k]:
                        diff += 1
                if diff == 1:
                    ans += 1
    return ans

for s, t, exp in examples:
    print(s, t, "->", count_brute(s, t), "(kỳ vọng", exp, ")")

aba baba -> 6 (kỳ vọng 6 )
ab bb -> 3 (kỳ vọng 3 )


**Độ phức tạp:** có `m·n` cặp điểm đầu, mỗi cặp tới `min(m,n)` độ dài, mỗi độ dài so sánh `L` ký tự ⇒ `O(m·n·min(m,n)²)`. Với `m = n = 100`, tổng số phép so sánh cỡ `Σ_{i,j} Σ_L L ≈ 1.7·10⁷` — vài giây trong Python, và tăng rất nhanh nếu ràng buộc lớn hơn.

**Nút thắt:** khi tăng `L` lên `L+1`, ta **so lại từ đầu** mọi ký tự đã so. Chỉ cần cộng thêm một ký tự mới!

## 6. Ý tưởng thuật toán / bước đột phá

Ba quan sát lần lượt loại bỏ phần thừa:

1. **Mở rộng dần:** cố định `(i, j)`, tăng `L` và chỉ so thêm `s[i+L−1]` với `t[j+L−1]`. Biến `diff` tăng dần; **nếu** `diff == 1` **thì** đếm; **nếu** `diff > 1` **thì** dừng (dài hơn chỉ tệ hơn). ⇒ `O(m·n·min(m,n))`.
2. **Lấy chỗ khác làm tâm:** mỗi cặp hợp lệ có **duy nhất** một vị trí khác `(i, j)` với `s[i] != t[j]`. Đếm theo vị trí đó: `l` = số cặp ký tự bằng nhau liên tiếp ngay **trước** `(i, j)` trên đường chéo, `r` = ngay **sau**. Số cặp nhận `(i, j)` làm chỗ khác = `(l + 1)(r + 1)`.
3. **Ghi nhớ `l`, `r`:** `l` chính là độ dài **hậu tố chung** của `s[:i]` và `t[:j]` (gọi `f[i][j]`), `r` là độ dài **tiền tố chung** của `s[i+1:]` và `t[j+1:]` (gọi `g[i+1][j+1]`). Hai bảng này tính bằng DP trong `O(m·n)`.

Và cuối cùng: duyệt **từng đường chéo** từ đầu tới cuối, với mỗi ô cuối `k` của đoạn, số đoạn hợp lệ kết thúc tại `k` = `last − prev`, với `last` là vị trí `x` gần nhất ≤ `k` và `prev` là vị trí `x` trước đó (mặc định `−1`). Chỉ cần 2 biến ⇒ `O(1)` bộ nhớ.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mở rộng dần từ mỗi cặp điểm đầu

Từng đoạn: hai vòng ngoài chọn `(i, j)`; vòng trong tăng `k` từ 0, cập nhật `diff`, đếm khi `diff == 1`, `break` khi `diff == 2`.

In [8]:
def count_extend(s, t):
    m, n = len(s), len(t)
    ans = 0
    for i in range(m):
        for j in range(n):
            diff = 0
            k = 0
            while i + k < m and j + k < n:
                if s[i + k] != t[j + k]:
                    diff += 1
                    if diff > 1:
                        break
                if diff == 1:
                    ans += 1          # xâu con độ dài k+1 hợp lệ
                k += 1
    return ans

for s, t, exp in examples:
    print(s, t, "->", count_extend(s, t), "(kỳ vọng", exp, ")")

aba baba -> 6 (kỳ vọng 6 )
ab bb -> 3 (kỳ vọng 3 )


Code nộp LeetCode cho cách 2 (`O(m·n·min(m,n))` ≈ `10⁶` bước, nộp được):

In [9]:
class SolutionExtend:
    def countSubstrings(self, s: str, t: str) -> int:
        m, n = len(s), len(t)
        ans = 0
        for i in range(m):
            for j in range(n):
                diff = 0
                for k in range(min(m - i, n - j)):
                    if s[i + k] != t[j + k]:
                        diff += 1
                        if diff == 2:
                            break
                    ans += diff        # diff là 0 hoặc 1 ở đây
        return ans

print([SolutionExtend().countSubstrings(s, t) for s, t, _ in examples])

[6, 3]


Mẹo `ans += diff`: trong vòng lặp `diff` chỉ là `0` hoặc `1` (khi lên `2` ta đã `break`), nên cộng `diff` chính là "đếm khi `diff == 1`".

**Độ phức tạp:** thời gian `O(m·n·min(m,n))`, bộ nhớ `O(1)`.

### 7.2 Cách 3 — Lấy ký tự khác nhau làm tâm, loang hai phía

Với mỗi `(i, j)` mà `s[i] != t[j]`:
- `l` = số bước lùi được sao cho `s[i−1−p] == t[j−1−p]` với mọi `p < l`;
- `r` = số bước tiến được sao cho `s[i+1+p] == t[j+1+p]`;
- cộng `(l + 1) * (r + 1)`.

In [10]:
def count_center(s, t):
    m, n = len(s), len(t)
    ans = 0
    for i in range(m):
        for j in range(n):
            if s[i] == t[j]:
                continue
            l = 0
            while i - l - 1 >= 0 and j - l - 1 >= 0 and s[i - l - 1] == t[j - l - 1]:
                l += 1
            r = 0
            while i + r + 1 < m and j + r + 1 < n and s[i + r + 1] == t[j + r + 1]:
                r += 1
            ans += (l + 1) * (r + 1)
    return ans

for s, t, exp in examples:
    print(s, t, "->", count_center(s, t), "(kỳ vọng", exp, ")")

aba baba -> 6 (kỳ vọng 6 )
ab bb -> 3 (kỳ vọng 3 )


In [11]:
class SolutionCenter:
    def countSubstrings(self, s: str, t: str) -> int:
        m, n = len(s), len(t)
        ans = 0
        for i in range(m):
            for j in range(n):
                if s[i] != t[j]:
                    l = 0
                    while i > l and j > l and s[i - l - 1] == t[j - l - 1]:
                        l += 1
                    r = 0
                    while i + r + 1 < m and j + r + 1 < n and s[i + r + 1] == t[j + r + 1]:
                        r += 1
                    ans += (l + 1) * (r + 1)
        return ans

print([SolutionCenter().countSubstrings(s, t) for s, t, _ in examples])

[6, 3]


**Độ phức tạp:** xấu nhất `O(m·n·min(m,n))` (ví dụ `s = "aaa…ab"`, `t = "aaa…a"` thì mỗi chỗ khác loang rất xa), bộ nhớ `O(1)`. Trung bình nhanh hơn cách 2 vì chỉ loang ở các chỗ khác.

### 7.3 Cách 4 — Quy hoạch động hai bảng `f`, `g`

- `f[i][j]` = độ dài hậu tố chung dài nhất của `s[:i]` và `t[:j]` (các ký tự bằng nhau liên tiếp kết thúc ngay trước `(i, j)`):
  `f[i][j] = f[i−1][j−1] + 1` nếu `s[i−1] == t[j−1]`, ngược lại `0`.
- `g[i][j]` = độ dài tiền tố chung dài nhất của `s[i:]` và `t[j:]`:
  `g[i][j] = g[i+1][j+1] + 1` nếu `s[i] == t[j]`, ngược lại `0`.
- Với mỗi `s[i] != t[j]`: `ans += (f[i][j] + 1) * (g[i+1][j+1] + 1)`.

In [12]:
def count_dp(s, t):
    m, n = len(s), len(t)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    g = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s[i - 1] == t[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if s[i] == t[j]:
                g[i][j] = g[i + 1][j + 1] + 1
    ans = 0
    for i in range(m):
        for j in range(n):
            if s[i] != t[j]:
                ans += (f[i][j] + 1) * (g[i + 1][j + 1] + 1)
    return ans

for s, t, exp in examples:
    print(s, t, "->", count_dp(s, t), "(kỳ vọng", exp, ")")

aba baba -> 6 (kỳ vọng 6 )
ab bb -> 3 (kỳ vọng 3 )


In thử hai bảng cho `s = "aba"`, `t = "baba"` để thấy ý nghĩa:

In [13]:
def show_tables(s, t):
    m, n = len(s), len(t)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    g = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s[i - 1] == t[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if s[i] == t[j]:
                g[i][j] = g[i + 1][j + 1] + 1
    print("f (hậu tố chung kết thúc trước (i,j)):")
    for row in f: print("   ", row)
    print("g (tiền tố chung bắt đầu tại (i,j)):")
    for row in g: print("   ", row)

show_tables("aba", "baba")

f (hậu tố chung kết thúc trước (i,j)):
    [0, 0, 0, 0, 0]
    [0, 0, 1, 0, 1]
    [0, 1, 0, 2, 0]
    [0, 0, 2, 0, 3]
g (tiền tố chung bắt đầu tại (i,j)):
    [0, 3, 0, 1, 0]
    [2, 0, 2, 0, 0]
    [0, 1, 0, 1, 0]
    [0, 0, 0, 0, 0]


**Độ phức tạp:** `O(m·n)` thời gian, `O(m·n)` bộ nhớ.

### 7.4 Cách 5 — Duyệt đường chéo, `O(1)` bộ nhớ

Trên một đường chéo, đánh số các ô `0, 1, 2, …`. Với mỗi ô `k` (coi là **điểm cuối** của đoạn), một đoạn `[a..k]` hợp lệ ⇔ chứa đúng một `x`. Gọi `last` = vị trí `x` gần nhất `≤ k`, `prev` = vị trí `x` trước `last` (mặc định cả hai là `−1`). Khi đó điểm đầu `a` phải thỏa `prev < a ≤ last` ⇒ có `last − prev` lựa chọn (nếu chưa có `x` nào thì `last = prev = −1` ⇒ 0).

Mỗi khi gặp `x`: `prev, last = last, k`. Mỗi ô: `ans += last − prev`.

In [14]:
def count_diagonal(s, t):
    m, n = len(s), len(t)
    ans = 0
    starts = [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]
    for i, j in starts:
        prev = last = -1
        k = 0
        while i < m and j < n:
            if s[i] != t[j]:
                prev, last = last, k
            ans += last - prev
            i += 1; j += 1; k += 1
    return ans

for s, t, exp in examples:
    print(s, t, "->", count_diagonal(s, t), "(kỳ vọng", exp, ")")

aba baba -> 6 (kỳ vọng 6 )
ab bb -> 3 (kỳ vọng 3 )


Một cách viết tương đương hay gặp: giữ `pre` = số ô `=` liên tiếp trước `x` gần nhất cộng 1, và `cur` = số ô `=` liên tiếp hiện tại cộng 1; mỗi ô cộng `pre` nếu đã có `x`. Cách `last − prev` dễ chứng minh hơn nên ta dùng nó.

**Độ phức tạp:** tổng số ô trên mọi đường chéo là `m·n` ⇒ thời gian `O(m·n)`, bộ nhớ `O(1)`.

## 8. TRACE từng bước

### 8.1 Trace cách 5 (đường chéo) trên `s = "aba"`, `t = "baba"`

Bảng mong đợi cho đường chéo bắt đầu `(0, 0)`: các cặp `(a,b), (b,a), (a,b)` đều khác ⇒ ký hiệu `xxx`.

| k | ô | ký hiệu | prev | last | cộng `last − prev` | ans tích lũy (đường này) |
|---|---|---|---|---|---|---|
| 0 | (0,0) a/b | x | −1 | 0 | 1 | 1 |
| 1 | (1,1) b/a | x | 0 | 1 | 1 | 2 |
| 2 | (2,2) a/b | x | 1 | 2 | 1 | 3 |

In [15]:
def trace_diagonal(s, t):
    m, n = len(s), len(t)
    ans = 0
    starts = [(0, j) for j in range(n)] + [(i, 0) for i in range(1, m)]
    for i0, j0 in starts:
        i, j = i0, j0
        prev = last = -1
        k = 0
        print(f"Đường chéo bắt đầu ({i0},{j0}):")
        while i < m and j < n:
            sym = '=' if s[i] == t[j] else 'x'
            if sym == 'x':
                prev, last = last, k
            ans += last - prev
            print(f"   k={k} ô=({i},{j}) {s[i]}/{t[j]} {sym}  prev={prev:>2} last={last:>2}  +{last - prev}  ans={ans}")
            i += 1; j += 1; k += 1
    print("Kết quả:", ans)
    return ans

trace_diagonal("aba", "baba")

Đường chéo bắt đầu (0,0):
   k=0 ô=(0,0) a/b x  prev=-1 last= 0  +1  ans=1
   k=1 ô=(1,1) b/a x  prev= 0 last= 1  +1  ans=2
   k=2 ô=(2,2) a/b x  prev= 1 last= 2  +1  ans=3
Đường chéo bắt đầu (0,1):
   k=0 ô=(0,1) a/a =  prev=-1 last=-1  +0  ans=3
   k=1 ô=(1,2) b/b =  prev=-1 last=-1  +0  ans=3
   k=2 ô=(2,3) a/a =  prev=-1 last=-1  +0  ans=3
Đường chéo bắt đầu (0,2):
   k=0 ô=(0,2) a/b x  prev=-1 last= 0  +1  ans=4
   k=1 ô=(1,3) b/a x  prev= 0 last= 1  +1  ans=5
Đường chéo bắt đầu (0,3):
   k=0 ô=(0,3) a/a =  prev=-1 last=-1  +0  ans=5
Đường chéo bắt đầu (1,0):
   k=0 ô=(1,0) b/b =  prev=-1 last=-1  +0  ans=5
   k=1 ô=(2,1) a/a =  prev=-1 last=-1  +0  ans=5
Đường chéo bắt đầu (2,0):
   k=0 ô=(2,0) a/b x  prev=-1 last= 0  +1  ans=6
Kết quả: 6


6

### 8.2 Trace cách 3 (tâm là chỗ khác) trên `s = "ab"`, `t = "bb"`

In [16]:
def trace_center(s, t):
    m, n = len(s), len(t)
    ans = 0
    for i in range(m):
        for j in range(n):
            if s[i] == t[j]:
                print(f"(i={i}, j={j}) {s[i]}=={t[j]} -> bỏ qua")
                continue
            l = 0
            while i > l and j > l and s[i - l - 1] == t[j - l - 1]:
                l += 1
            r = 0
            while i + r + 1 < m and j + r + 1 < n and s[i + r + 1] == t[j + r + 1]:
                r += 1
            ans += (l + 1) * (r + 1)
            print(f"(i={i}, j={j}) {s[i]}!={t[j]} -> l={l}, r={r}, cộng ({l}+1)*({r}+1)={(l + 1) * (r + 1)}, ans={ans}")
    return ans

trace_center("ab", "bb")

(i=0, j=0) a!=b -> l=0, r=1, cộng (0+1)*(1+1)=2, ans=2
(i=0, j=1) a!=b -> l=0, r=0, cộng (0+1)*(0+1)=1, ans=3
(i=1, j=0) b==b -> bỏ qua
(i=1, j=1) b==b -> bỏ qua


3

### 8.3 Trace trên ví dụ dài hơn (tự dựng): `s = "computer"`, `t = "computation"`

Ví dụ này **tự dựng** (lấy từ câu mô tả của đề) — kết quả được kiểm chứng bằng brute force; nên đối chiếu lại đề gốc nếu muốn. Ta chỉ in các chỗ khác cho đóng góp lớn:

In [17]:
s, t = "computer", "computation"
m, n = len(s), len(t)
big = []
for i in range(m):
    for j in range(n):
        if s[i] != t[j]:
            l = 0
            while i > l and j > l and s[i - l - 1] == t[j - l - 1]:
                l += 1
            r = 0
            while i + r + 1 < m and j + r + 1 < n and s[i + r + 1] == t[j + r + 1]:
                r += 1
            if (l + 1) * (r + 1) >= 4:
                big.append((i, j, s[i], t[j], l, r, (l + 1) * (r + 1)))
for row in big:
    print("i=%d j=%d %s/%s l=%d r=%d -> %d" % row)
print("Tổng (cách 3):", count_center(s, t), "| brute:", count_brute(s, t))

i=6 j=6 e/a l=6 r=0 -> 7
Tổng (cách 3): 90 | brute: 90


Đóng góp lớn nhất `(i=6, j=6) e/a` với `l=6` (`"comput"` bằng nhau ở bên trái), `r=0` ⇒ `7` cặp: `"e"/"a"`, `"te"/"ta"`, …, `"compute"/"computa"` — đúng ví dụ trong đề.

## 9. Code nộp LeetCode

Lời giải chính: duyệt đường chéo, `O(m·n)` thời gian, `O(1)` bộ nhớ.

In [18]:
class Solution:
    def countSubstrings(self, s: str, t: str) -> int:
        m, n = len(s), len(t)
        ans = 0
        for d in range(-(m - 1), n):          # d = j - i
            i, j = max(0, -d), max(0, d)
            prev = last = -1
            k = 0
            while i < m and j < n:
                if s[i] != t[j]:
                    prev, last = last, k
                ans += last - prev
                i += 1
                j += 1
                k += 1
        return ans

for s, t, exp in examples:
    assert Solution().countSubstrings(s, t) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Song ánh cặp ↔ đoạn chéo:** cặp `(s[i..i+L), t[j..j+L))` tương ứng 1–1 với đoạn ô `(i+p, j+p), p = 0..L−1` trên đường chéo `j − i = d`. Khoảng cách Hamming của cặp = số ô `x` trong đoạn. Vậy đáp án = tổng trên mọi đường chéo của số đoạn liên tục chứa **đúng một** `x`.

**Cách 3 / cách 4 (đếm theo `x`):** mỗi đoạn có đúng một `x` được đếm **đúng một lần** — tại chính ô `x` đó. Với ô `x` cố định ở vị trí `k`, đoạn `[a..b]` chứa nó và không chứa `x` khác ⇔ `k − l ≤ a ≤ k` và `k ≤ b ≤ k + r` với `l`, `r` là số ô `=` liên tiếp bên trái/phải. Hai lựa chọn độc lập ⇒ `(l+1)(r+1)`. Trong cách 4, `l = f[i][j]` và `r = g[i+1][j+1]` đúng theo định nghĩa của hai bảng.

**Cách 5 (đếm theo điểm cuối):** mỗi đoạn được đếm đúng một lần — tại điểm cuối `b`. Với `b` cố định, gọi `last` là `x` cuối cùng trong `[0..b]`, `prev` là `x` ngay trước. Đoạn `[a..b]` chứa đúng một `x` ⇔ nó chứa `last` mà không chứa `prev` ⇔ `prev < a ≤ last` ⇒ đúng `last − prev` lựa chọn. Nếu chưa có `x` nào, `last = prev = −1` cho 0 lựa chọn — đúng. Bất biến "`last`, `prev` là hai `x` cuối cùng" được giữ vì ta chỉ cập nhật `prev, last = last, k` khi gặp `x` mới.

## 11. Phân tích Time / Space Complexity

Gọi `m = len(s)`, `n = len(t)`, `L = min(m, n)`.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(m·n·L²)` | `O(1)` |
| 2. Mở rộng dần | `O(m·n·L)` | `O(1)` |
| 3. Tâm là chỗ khác | `O(m·n·L)` xấu nhất, thường nhanh hơn | `O(1)` |
| 4. DP `f`, `g` | `O(m·n)` | `O(m·n)` |
| 5. Đường chéo | `O(m·n)` | `O(1)` |

Với `m = n = 100`: cách 4, 5 chỉ ~`10⁴` bước; cách 2, 3 ~`10⁶`; brute ~`1.7·10⁷` (vài giây; ta chỉ chạy cỡ 40). Đo thử (brute chỉ chạy cỡ 40):

In [19]:
rnd = random.Random(1638)
s = ''.join(rnd.choice("ab") for _ in range(100))
t = ''.join(rnd.choice("ab") for _ in range(100))
for name, fn in [("extend", count_extend), ("center", count_center), ("dp", count_dp), ("diagonal", count_diagonal)]:
    t0 = time.perf_counter(); r = fn(s, t); dt = time.perf_counter() - t0
    print(f"{name:<9} m=n=100: {r:>6}  {dt * 1000:7.1f} ms")
s40, t40 = s[:40], t[:40]
t0 = time.perf_counter(); r = count_brute(s40, t40); dt = time.perf_counter() - t0
print(f"brute     m=n=40 : {r:>6}  {dt * 1000:7.1f} ms  (so diagonal: {count_diagonal(s40, t40)})")

extend    m=n=100:  19259      3.6 ms
center    m=n=100:  19259      2.0 ms
dp        m=n=100:  19259      1.3 ms
diagonal  m=n=100:  19259      0.7 ms
brute     m=n=40 :   2845      9.1 ms  (so diagonal: 2845)


## 12. Bộ test

Gồm: ví dụ đề; edge case (độ dài 1, hai xâu giống hệt, hai xâu không có ký tự chung, xâu toàn một chữ); ví dụ tự dựng có đáp án tính bằng công thức (kiểm chứng bằng brute force); và đối chiếu ngẫu nhiên.

In [20]:
ALL = {
    "brute": count_brute,
    "extend": count_extend,
    "center": count_center,
    "dp": count_dp,
    "diagonal": count_diagonal,
    "SolExtend": lambda s, t: SolutionExtend().countSubstrings(s, t),
    "SolCenter": lambda s, t: SolutionCenter().countSubstrings(s, t),
    "Solution": lambda s, t: Solution().countSubstrings(s, t),
}

def check(s, t, exp=None):
    res = {name: fn(s, t) for name, fn in ALL.items()}
    assert len(set(res.values())) == 1, (s, t, res)
    if exp is not None:
        assert res["Solution"] == exp, (s, t, res, exp)
    return res["Solution"]

for s, t, exp in examples:
    check(s, t, exp)
print("Ví dụ đề: OK")

# Edge case (tự dựng, đáp án có lý do rõ ràng, đã kiểm chứng bằng brute force trong check)
check("a", "a", 0)            # giống nhau, không có cặp nào khác đúng 1
check("a", "b", 1)
check("abc", "abc", 6)        # tự dựng: mỗi chỗ khác (i!=j) ... kiểm chứng bằng brute
check("aaa", "bbb", 9)        # mọi cặp ký tự đơn khác nhau: 3*3; xâu dài hơn có >=2 chỗ khác
check("aaaa", "a", 0)
check("ab", "ba", check("ba", "ab"))   # tính đối xứng
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


Giải thích `check("abc", "abc", 6)`: ví dụ tự dựng. Các ô `x` là `(i, j)` với `i ≠ j` (vì 3 ký tự khác nhau). Mỗi ô `x` có `l = r = 0` (hàng xóm trên đường chéo cũng là `x` hoặc ra ngoài) ⇒ mỗi ô đóng góp 1; có `9 − 3 = 6` ô ⇒ `6`. Kết quả cũng được brute force xác nhận (đối chiếu lại đề gốc nếu muốn).

Công thức đặc biệt: nếu `s = c·…·c` (m lần) và `t = d·…·d` (n lần) với `c ≠ d`, mọi ô là `x` ⇒ chỉ các đoạn độ dài 1 hợp lệ ⇒ đáp án `m·n`.

In [21]:
for m in range(1, 8):
    for n in range(1, 8):
        assert check("a" * m, "b" * n) == m * n
# Một chữ khác ở giữa: s = 'a'*m, t = 'a'*n với một vị trí đổi thành 'b'
for m in range(1, 7):
    for n in range(1, 7):
        for p in range(n):
            t = "a" * p + "b" + "a" * (n - p - 1)
            check("a" * m, t)
print("Họ xâu có cấu trúc: OK")

rnd = random.Random(2024)
for _ in range(1500):
    s = ''.join(rnd.choice("abc"[:rnd.randint(1, 3)]) for _ in range(rnd.randint(1, 9)))
    t = ''.join(rnd.choice("abc"[:rnd.randint(1, 3)]) for _ in range(rnd.randint(1, 9)))
    check(s, t)
print("1500 cặp ngẫu nhiên nhỏ: OK")

for _ in range(20):
    s = ''.join(rnd.choice("ab") for _ in range(rnd.randint(1, 100)))
    t = ''.join(rnd.choice("ab") for _ in range(rnd.randint(1, 100)))
    assert count_extend(s, t) == count_center(s, t) == count_dp(s, t) == count_diagonal(s, t) == Solution().countSubstrings(s, t)
print("20 cặp cỡ lớn (đến 100), so 5 cách nhanh: OK")

Họ xâu có cấu trúc: OK
1500 cặp ngẫu nhiên nhỏ: OK


20 cặp cỡ lớn (đến 100), so 5 cách nhanh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đếm theo **nội dung** (dùng `set`) thay vì theo **vị trí**

In [22]:
def count_bug_set(s, t):
    subs_s = {s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)}
    subs_t = {t[i:j] for i in range(len(t)) for j in range(i + 1, len(t) + 1)}
    ans = 0
    for a in subs_s:
        for b in subs_t:
            if len(a) == len(b) and sum(x != y for x, y in zip(a, b)) == 1:
                ans += 1
    return ans

print("sai :", count_bug_set("aba", "baba"), "| đúng:", Solution().countSubstrings("aba", "baba"))

sai : 2 | đúng: 6


**Vì sao sai:** đề đếm **cặp vị trí** — `"a"` ở vị trí 0 và 2 của `s` là hai xâu con khác nhau (ví dụ 1 có 6 cặp, set chỉ cho 2). **Sửa:** duyệt theo chỉ số, không gộp xâu trùng nội dung.

### Lỗi 2 — Đếm cả cặp **giống hệt nhau** (`diff ≤ 1` thay vì `== 1`)

In [23]:
def count_bug_le1(s, t):
    m, n = len(s), len(t)
    ans = 0
    for i in range(m):
        for j in range(n):
            diff = 0
            for k in range(min(m - i, n - j)):
                diff += s[i + k] != t[j + k]
                if diff > 1:
                    break
                ans += 1                      # SAI: cộng cả khi diff == 0
    return ans

print("sai :", count_bug_le1("ab", "bb"), "| đúng:", Solution().countSubstrings("ab", "bb"))

sai : 5 | đúng: 3


**Vì sao sai:** đề yêu cầu thay ký tự bằng một ký tự **khác** ⇒ khác **đúng** một vị trí; cặp `("b", "b")` không hợp lệ. **Sửa:** chỉ cộng khi `diff == 1` (hoặc `ans += diff` khi `diff ∈ {0, 1}`).

### Lỗi 3 — Công thức tâm viết `l * r` hoặc `l + r + 1`

In [24]:
def count_bug_formula(s, t):
    m, n = len(s), len(t)
    ans = 0
    for i in range(m):
        for j in range(n):
            if s[i] != t[j]:
                l = 0
                while i > l and j > l and s[i - l - 1] == t[j - l - 1]:
                    l += 1
                r = 0
                while i + r + 1 < m and j + r + 1 < n and s[i + r + 1] == t[j + r + 1]:
                    r += 1
                ans += l + r + 1                 # SAI: phải là (l+1)*(r+1)
    return ans

s, t = "xay", "xby"
print("sai :", count_bug_formula(s, t), "| đúng:", Solution().countSubstrings(s, t), "| brute:", count_brute(s, t))

sai : 9 | đúng: 10 | brute: 10


**Vì sao sai:** điểm đầu (`l + 1` lựa chọn) và điểm cuối (`r + 1` lựa chọn) được chọn **độc lập** ⇒ **nhân**, không cộng. Với `"xay"`/`"xby"` có 4 đoạn chứa `a/b`: `a`, `xa`, `ay`, `xay`. **Sửa:** `ans += (l + 1) * (r + 1)`.

### Lỗi 4 — DP dùng nhầm `g[i][j]` thay vì `g[i+1][j+1]`

In [25]:
def count_bug_dp_index(s, t):
    m, n = len(s), len(t)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    g = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s[i - 1] == t[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
    for i in range(m - 1, -1, -1):
        for j in range(n - 1, -1, -1):
            if s[i] == t[j]:
                g[i][j] = g[i + 1][j + 1] + 1
    ans = 0
    for i in range(m):
        for j in range(n):
            if s[i] != t[j]:
                ans += (f[i][j] + 1) * (g[i][j] + 1)     # SAI: g[i][j] = 0 vì s[i] != t[j]
    return ans

print("sai :", count_bug_dp_index("xay", "xby"), "| đúng:", count_dp("xay", "xby"))

sai : 8 | đúng: 10


**Vì sao sai:** tại ô `x`, `g[i][j]` luôn bằng `0` (vì `s[i] != t[j]`), nên phần bên phải bị bỏ mất. Cần tiền tố chung **sau** ô `x`: `g[i+1][j+1]`. Tương tự `f[i][j]` (không phải `f[i+1][j+1]`) là hậu tố chung **trước** ô `x` — vì `f` được đánh chỉ số lệch 1.

**Lỗi phụ hay gặp:** tạo bảng bằng `[[0]*(n+1)]*(m+1)` (xem mục 3.3) — mọi hàng dùng chung một list.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "So **cặp xâu con cùng độ dài** của hai xâu" ⇒ nghĩ ngay tới **lưới `m × n`** và **đường chéo** (giống Longest Common Substring).
- "Khác **đúng một** chỗ" ⇒ đếm theo **chính chỗ khác đó** (tâm) và nhân số lựa chọn hai bên: `(l+1)(r+1)`.
- "Đếm số đoạn con có đúng `k` phần tử đặc biệt" ⇒ giữ vị trí các phần tử đặc biệt gần nhất (`last − prev`), hoặc dùng `atMost(k) − atMost(k−1)`.
- Ràng buộc `≤ 100` cho phép `O(n³)`; nhưng nêu được `O(n²)` sẽ ghi điểm khi phỏng vấn.
- Luôn viết brute force nhỏ để kiểm, vì các công thức đếm rất dễ lệch 1.

## 15. Biến thể của bài

1. **Khác nhau tối đa `k` ký tự** (thay vì đúng 1): trên mỗi đường chéo dùng **cửa sổ trượt** giữ tối đa `k` ô `x` ⇒ `O(m·n)`.
2. **Khác nhau đúng `k` ký tự:** `atMost(k) − atMost(k−1)`, hoặc giữ hàng đợi vị trí các `x`.
3. **LeetCode 718 — Maximum Length of Repeated Subarray / Longest Common Substring:** chính là bảng `f` (hậu tố chung) — lấy max.
4. **Đếm cặp xâu con giống hệt nhau** (`diff = 0`): tổng `f[i][j]` trên mọi ô.
5. **Chỉ đếm xâu con phân biệt của `s`** có một xâu "hàng xóm" trong `t` — cần băm xâu/ trie, khó hơn hẳn.
6. **LeetCode 1790 / 859 — Buddy Strings:** kiểm tra hai xâu khác nhau đúng hai vị trí có thể hoán đổi.

In [26]:
def count_at_most_k(s, t, k):
    """Số cặp xâu con cùng độ dài khác nhau <= k vị trí (cửa sổ trượt trên đường chéo)."""
    m, n = len(s), len(t)
    ans = 0
    for d in range(-(m - 1), n):
        i, j = max(0, -d), max(0, d)
        cells = []
        while i < m and j < n:
            cells.append(s[i] != t[j]); i += 1; j += 1
        left = bad = 0
        for right, isx in enumerate(cells):
            bad += isx
            while bad > k:
                bad -= cells[left]; left += 1
            ans += right - left + 1
    return ans

def count_exact_k(s, t, k):
    return count_at_most_k(s, t, k) - (count_at_most_k(s, t, k - 1) if k > 0 else 0)

print("đúng 1 (aba, baba):", count_exact_k("aba", "baba", 1))
print("đúng 0 (aba, baba):", count_exact_k("aba", "baba", 0))
print("đúng 2 (aba, baba):", count_exact_k("aba", "baba", 2))

đúng 1 (aba, baba): 6
đúng 0 (aba, baba): 10
đúng 2 (aba, baba): 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | so mọi cặp (i, j, L) | `O(m·n·L²)` | `O(1)` | kiểm chứng trên input nhỏ |
| 2. Mở rộng dần | cố định điểm đầu, tăng L, dừng khi diff = 2 | `O(m·n·L)` | `O(1)` | đủ nhanh với n ≤ 100, dễ viết |
| 3. Tâm là chỗ khác | loang trái/phải, `(l+1)(r+1)` | `O(m·n·L)` xấu nhất | `O(1)` | trực giác tốt, dễ giải thích |
| 4. DP `f`, `g` | ghi nhớ l, r | `O(m·n)` | `O(m·n)` | khi cần tối ưu và muốn công thức rõ |
| 5. Đường chéo | `last − prev` cho mỗi điểm cuối | `O(m·n)` | `O(1)` | tối ưu nhất, mở rộng được sang "≤ k" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().countSubstrings("abc", "abd")` bằng bao nhiêu? Thử đoán trước khi chạy.

**Bài 2 (tính tay):** Với dãy ký hiệu trên một đường chéo `"=x==x="`, có bao nhiêu đoạn con chứa đúng một `x`? Tính bằng `(l+1)(r+1)` cho từng `x` **và** bằng `last − prev` cho từng điểm cuối.

**Bài 3 (điền TODO):** Hoàn thiện `count_equal_pairs(s, t)` đếm số cặp xâu con cùng độ dài **giống hệt nhau** trong `O(m·n)`.

**Bài 4 (tự viết lại):** Viết lại cách 4 (DP) nhưng chỉ dùng **một** bảng `f` và tính phần bên phải bằng bảng `g` thu gọn **một chiều** (gợi ý: duyệt ngược theo `i`, giữ `g` của hàng `i+1`).

**Bài 5 (mini challenge):** Viết `count_exactly_two(s, t)` đếm số cặp xâu con cùng độ dài khác nhau **đúng 2** vị trí trong `O(m·n)` bằng duyệt đường chéo (giữ ba vị trí `x` gần nhất).

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_equal_pairs(s, t):
    m, n = len(s), len(t)
    # TODO: f[i][j] = độ dài hậu tố chung của s[:i], t[:j]; đáp án = tổng mọi f[i][j]
    pass

# Bài 4 — TODO
def count_dp_1d(s, t):
    # TODO: tính f đầy đủ, nhưng g chỉ lưu hàng i+1 khi duyệt i giảm dần
    pass

# Bài 5 — TODO
def count_exactly_two(s, t):
    # TODO: trên mỗi đường chéo giữ p3 < p2 < p1 là ba vị trí 'x' gần nhất
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Vẽ bảng `=`/`x` 3×3.
- Mức 2: Đường chéo chính là `==x` — ô `x` có `l = 2`, `r = 0`.
- Mức 3: Cộng thêm các ô `x` ngoài đường chéo chính (mỗi ô đóng góp bao nhiêu?).

**Bài 2**
- Mức 1: Hai `x` ở vị trí 1 và 4.
- Mức 2: `x`@1 có `l = 1`, `r = 2`; `x`@4 có `l = 2`, `r = 1`.
- Mức 3: `2·3 + 3·2 = 12`; kiểm tra lại bằng `last − prev` cộng dồn từng ô.

**Bài 3**
- Mức 1: Một cặp giống nhau = đoạn chéo chỉ có `=`.
- Mức 2: Số đoạn `=` kết thúc tại `(i−1, j−1)` bằng `f[i][j]`.
- Mức 3: `sum(map(sum, f))`.

**Bài 4**
- Mức 1: `g[i][j]` chỉ phụ thuộc `g[i+1][j+1]`.
- Mức 2: Duyệt `i` từ `m−1` về `0`, giữ `nxt = g[i+1]`, tính `cur = g[i]`.
- Mức 3: Ô `x` tại `(i, j)` dùng `nxt[j+1]`.

**Bài 5**
- Mức 1: Đoạn kết thúc tại `k` chứa đúng 2 `x` ⇔ chứa `p2` nhưng không chứa `p3`.
- Mức 2: Số điểm đầu = `p2 − p3`.
- Mức 3: Khởi tạo `p1 = p2 = p3 = −1`; gặp `x`: `p3, p2, p1 = p2, p1, k`.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1
ans1 = Solution().countSubstrings("abc", "abd")
print("Bài 1:", ans1)
assert ans1 == count_brute("abc", "abd")

# Bài 2
sym = "=x==x="
way1 = 0
for k, ch in enumerate(sym):
    if ch == 'x':
        l = 0
        while k - l - 1 >= 0 and sym[k - l - 1] == '=':
            l += 1
        r = 0
        while k + r + 1 < len(sym) and sym[k + r + 1] == '=':
            r += 1
        way1 += (l + 1) * (r + 1)
prev = last = -1
way2 = 0
for k, ch in enumerate(sym):
    if ch == 'x':
        prev, last = last, k
    way2 += last - prev
brute2 = sum(sym[a:b].count('x') == 1 for a in range(len(sym)) for b in range(a + 1, len(sym) + 1))
print("Bài 2:", way1, way2, brute2)
assert way1 == way2 == brute2 == 12

Bài 1: 9
Bài 2: 12 12 12


In [29]:
# Bài 3
def count_equal_pairs_sol(s, t):
    m, n = len(s), len(t)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    total = 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s[i - 1] == t[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
                total += f[i][j]
    return total

def count_equal_brute(s, t):
    return sum(s[i:i + L] == t[j:j + L]
               for i in range(len(s)) for j in range(len(t))
               for L in range(1, min(len(s) - i, len(t) - j) + 1))

# Bài 4
def count_dp_1d_sol(s, t):
    m, n = len(s), len(t)
    f = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s[i - 1] == t[j - 1]:
                f[i][j] = f[i - 1][j - 1] + 1
    nxt = [0] * (n + 1)          # g[i+1][*]
    ans = 0
    for i in range(m - 1, -1, -1):
        cur = [0] * (n + 1)      # g[i][*]
        for j in range(n - 1, -1, -1):
            if s[i] == t[j]:
                cur[j] = nxt[j + 1] + 1
            else:
                ans += (f[i][j] + 1) * (nxt[j + 1] + 1)
        nxt = cur
    return ans

rnd = random.Random(77)
for _ in range(800):
    s = ''.join(rnd.choice("ab") for _ in range(rnd.randint(1, 8)))
    t = ''.join(rnd.choice("ab") for _ in range(rnd.randint(1, 8)))
    assert count_equal_pairs_sol(s, t) == count_equal_brute(s, t)
    assert count_dp_1d_sol(s, t) == count_brute(s, t)
print("Bài 3, Bài 4: OK")

Bài 3, Bài 4: OK


In [30]:
# Bài 5
def count_exactly_two_sol(s, t):
    m, n = len(s), len(t)
    ans = 0
    for d in range(-(m - 1), n):
        i, j = max(0, -d), max(0, d)
        p1 = p2 = p3 = -1
        k = 0
        while i < m and j < n:
            if s[i] != t[j]:
                p3, p2, p1 = p2, p1, k
            ans += p2 - p3
            i += 1; j += 1; k += 1
    return ans

def count_exactly_two_brute(s, t):
    return sum(sum(a != b for a, b in zip(s[i:i + L], t[j:j + L])) == 2
               for i in range(len(s)) for j in range(len(t))
               for L in range(1, min(len(s) - i, len(t) - j) + 1))

rnd = random.Random(88)
for _ in range(800):
    s = ''.join(rnd.choice("abc") for _ in range(rnd.randint(1, 8)))
    t = ''.join(rnd.choice("abc") for _ in range(rnd.randint(1, 8)))
    assert count_exactly_two_sol(s, t) == count_exactly_two_brute(s, t) == count_exact_k(s, t, 2)
print("Bài 5: OK | ví dụ aba/baba:", count_exactly_two_sol("aba", "baba"))

Bài 5: OK | ví dụ aba/baba: 3


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "duyệt mọi đường chéo của lưới m × n":**

```python
for d in range(-(m - 1), n):          # d = j - i
    i, j = max(0, -d), max(0, d)
    # khởi tạo trạng thái của đường chéo
    while i < m and j < n:
        # xử lý ô (i, j)
        i += 1; j += 1
```

**Template "đếm đoạn con chứa đúng một phần tử đặc biệt":**

```python
prev = last = -1
for k, special in enumerate(seq):
    if special:
        prev, last = last, k
    ans += last - prev
```

**Template "tâm":** `ans += (l + 1) * (r + 1)` với `l`, `r` là số phần tử "bình thường" liên tiếp hai bên.

**Checklist:**
- [ ] Đếm theo **vị trí** (cặp chỉ số), không gộp xâu trùng nội dung?
- [ ] Chỉ đếm cặp khác **đúng 1** (không tính cặp giống hệt)?
- [ ] Công thức nhân `(l+1)(r+1)`, không cộng?
- [ ] Chỉ số DP: `f[i][j]` (trước ô) và `g[i+1][j+1]` (sau ô)?
- [ ] Bảng 2D tạo bằng list comprehension, không nhân list?
- [ ] Đã so với brute force trên vài trăm cặp xâu ngẫu nhiên nhỏ?
- [ ] Chữ ký đúng: `def countSubstrings(self, s: str, t: str) -> int`?